# Encrypted Vector Store with LangChain-enVector

This example shows `Envector`, a LangChain vector store that keeps documents and
their embeddings in an enVector index. The embeddings are stored encrypted, and
the server scores a query against them without decrypting anything; with
metadata encryption the document text and metadata are encrypted as well.

In this example, we'll see:

- How to add LangChain `Document`s and search them with a score
- How to narrow a search by metadata
- How to read documents back by ID, replace one in place and delete one
- How to use the store as a retriever in a small RAG chain

## Prerequisites
- enVector server reachable from this notebook environment
- A key path for the key this notebook registers (it is created if missing)
- Ollama running locally, with internet access to pull the two models used below
- `langchain-envector` and `langchain-ollama` packages installed

### Install packages

`langchain-envector` provides the vector store; `langchain-ollama` lets LangChain use
the models served by Ollama.

In [ ]:
# !pip install langchain-envector langchain-ollama

## Configure connection and models
Fill in your enVector connection details. The embedding model turns documents and
queries into vectors; the LLM is used only in the last section, to answer a
question from the retrieved documents. Both run locally through Ollama here, so
no API key is needed.

In [ ]:
import os

address = os.getenv("ENVECTOR_ADDRESS", "0.0.0.0:50050")
token = os.getenv("ENVECTOR_ACCESS_TOKEN", "")

key_path = "./keys"
key_id = "vectorstore_key"
index_name = "vectorstore_demo"
embedding_model = "all-minilm:l6-v2"
llm_model = "gemma3:270m"

print(f"Using key '{key_id}'")
print(f"Using index '{index_name}'")
print(f"Embedding model: {embedding_model}")
print(f"LLM: {llm_model}")

## Pull the Ollama models
Ollama downloads a model on first use; pulling them up front keeps the cells below short.

In [ ]:
!ollama pull {embedding_model}
!ollama pull {llm_model}

## Set up the embedding model
The embedding model's output size becomes the index dimension. Scores are inner
products computed under encryption, so the embeddings must be unit norm; Ollama
returns them normalized.

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=embedding_model)
vector_dim = len(embeddings.embed_query("dimension probe"))
print(f"Embedding dimension: {vector_dim}")

## Create the vector store

`Envector` takes an `EnvectorConfig`: where the server is, which key encrypts the
data, and how the index is set up. `create_if_missing=True` creates the index on
first use, and `IndexSettings.metadata_encryption=True` encrypts the document text
and metadata along with the vectors.

In [ ]:
from langchain_envector import Envector
from langchain_envector.config import ConnectionConfig, EnvectorConfig, IndexSettings, KeyConfig

config = EnvectorConfig(
    connection=ConnectionConfig(address=address, access_token=token) if token else ConnectionConfig(address=address),
    key=KeyConfig(key_path=key_path, key_id=key_id),
    index=IndexSettings(index_name=index_name, dim=vector_dim, metadata_encryption=True),
    create_if_missing=True,
)
store = Envector(config=config, embeddings=embeddings)

## Add documents

A `Document` has `page_content`, which is embedded, and `metadata`, which is stored
next to it and can narrow a search later. `add_documents` returns the item IDs the
server assigned; keep them to read, update or delete these documents afterwards.

In [ ]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="The Eiffel Tower in Paris was completed in 1889 for the World's Fair.",
        metadata={"topic": "landmarks", "country": "France"},
    ),
    Document(
        page_content="The Colosseum in Rome could hold about fifty thousand spectators.",
        metadata={"topic": "landmarks", "country": "Italy"},
    ),
    Document(
        page_content="Croissants are a buttery, flaky pastry eaten for breakfast in France.",
        metadata={"topic": "food", "country": "France"},
    ),
    Document(
        page_content="Carbonara is a Roman pasta dish made with eggs, pecorino and guanciale.",
        metadata={"topic": "food", "country": "Italy"},
    ),
    Document(
        page_content="The Louvre in Paris is the most visited art museum in the world.",
        metadata={"topic": "landmarks", "country": "France"},
    ),
]

ids = store.add_documents(documents)
print(ids)

## Search with a score

`similarity_search_with_score` embeds the query, the server scores it against the
encrypted vectors, and the best `k` documents come back with their scores. The
score is the inner product of two unit vectors: `1` is an identical vector, and
higher is closer.

In [ ]:
query = "Which pastry do people in France eat in the morning?"

for doc, score in store.similarity_search_with_score(query, k=3):
    print(f"[{score:.3f}] {doc.page_content}  {doc.metadata}")

## Narrow a search by metadata

`filter` keeps only the hits whose metadata has the given values. It is applied
after the server returns its best hits, so `fetch_k` asks for more than `k` to
still have `k` documents left after filtering.

In [ ]:
query = "What is there to see in Paris?"

for doc in store.similarity_search(query, k=2, filter={"topic": "landmarks"}, fetch_k=10):
    print(f"{doc.page_content}  {doc.metadata}")

## Read documents back by ID

`get_by_ids` returns the documents for the given item IDs without a search. A search
result's `doc.id` is the same item ID.

In [ ]:
for doc in store.get_by_ids(ids[:2]):
    print(f"{doc.id}: {doc.page_content}")

## Replace a document in place

`update_documents` replaces a document's vector and stored content while keeping
its ID. The new `page_content` is embedded again, so the document now matches
different queries. IDs that match no row are reported in `not_found_item_ids`
rather than raised.

In [ ]:
result = store.update_documents(
    [ids[2]],
    [
        Document(
            page_content="Pain au chocolat is a French pastry with a bar of chocolate inside.",
            metadata={"topic": "food", "country": "France"},
        )
    ],
)
print(result)

for doc, score in store.similarity_search_with_score("a pastry with chocolate", k=1):
    print(f"[{score:.3f}] {doc.id}: {doc.page_content}")

## Delete a document

`delete` takes a list of item IDs. It waits for the server by default, and the
document is gone from `get_by_ids` as soon as it returns.

In [ ]:
store.delete([ids[1]])

print(store.get_by_ids([ids[1]]))

## Use the store as a retriever

A retriever is LangChain's name for anything that takes a question and returns the
documents that match it. `as_retriever` turns the store into one: each question
runs `similarity_search` with the `search_kwargs` given here.

Below, the two closest documents become the context a small LLM answers from,
the usual RAG pattern. First the pieces:

- `retriever` finds the two documents closest to the question.
- `prompt` is a template with two blanks, `{context}` and `{question}`.
- `llm` writes the answer.
- `format_docs` joins the documents into one string, since the template takes text.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

retriever = store.as_retriever(search_kwargs={"k": 2})
prompt = PromptTemplate.from_template(
    "Answer the question using only the context.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer:"
)
llm = OllamaLLM(model=llm_model, temperature=0)


def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

### Connect the pieces with LCEL

The chain below is written in LCEL, the LangChain Expression Language. Read `|`
as "then": what the left side returns is handed to the right side, like a pipe
in a shell. Writing the chain runs nothing yet; it runs when
`chain.invoke(question)` is called.

For the question "When was the Eiffel Tower completed?", the steps are:

1. The dict at the start sends the same question down two paths and collects
   the results under its keys:
   - `"context"`: `retriever | format_docs` searches with the question, then
     joins the two documents found into one string.
   - `"question"`: `RunnablePassthrough()` passes the question on unchanged.

   The result is `{"context": "<the two documents>", "question": "When was the Eiffel Tower completed?"}`.
2. `prompt` fills its two blanks with those two values.
3. `llm` reads the filled-in prompt and writes the answer.

The chain does the same as this plain function:

```python
def answer(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    filled_prompt = prompt.invoke({"context": context, "question": question})
    return llm.invoke(filled_prompt)
```

Written as a chain, it can also be streamed, batched or run asynchronously
(`chain.stream`, `chain.batch`, `chain.ainvoke`) with no further code.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm
print(chain.invoke("When was the Eiffel Tower completed?"))

### Clean Up

We can delete the created index and the registered key when they are no longer needed.

In [ ]:
store.client.ev.drop_index(index_name)

In [ ]:
store.client.ev.delete_key(key_id)